# 🤖 Model Training — Traffic Speed Prediction

This notebook trains ML models to predict traffic speed at monitoring points.

**Features:** time-of-day, day-of-week, rush hour, weather, lag speeds
**Target:** `current_speed` (km/h)
**Models:** RandomForest, GradientBoosting

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text
from src.database.connection import engine
from src.config.settings import settings

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import joblib

print('✅ All imports loaded')

✅ All imports loaded


## 1. Load Data from PostGIS

In [2]:
# Load all traffic readings
traffic_df = pd.read_sql(
    'SELECT * FROM traffic_readings ORDER BY timestamp',
    engine
)
traffic_df['timestamp'] = pd.to_datetime(traffic_df['timestamp'])

# Load weather readings
weather_df = pd.read_sql(
    'SELECT * FROM weather_readings ORDER BY timestamp',
    engine
)
weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])

print(f'Traffic readings: {len(traffic_df):,}')
print(f'Weather readings: {len(weather_df):,}')
print(f'Traffic time range: {traffic_df["timestamp"].min()} → {traffic_df["timestamp"].max()}')
print(f'Locations: {traffic_df["location_name"].nunique()}')
print(f'\nTraffic columns: {list(traffic_df.columns)}')

Traffic readings: 289
Weather readings: 15
Traffic time range: 2026-04-18 16:24:47.076680 → 2026-04-18 18:53:30.701281
Locations: 9

Traffic columns: ['id', 'timestamp', 'location_name', 'lat', 'lon', 'current_speed', 'free_flow_speed', 'confidence', 'congestion_ratio', 'geometry']


## 2. Feature Engineering

In [3]:
df = traffic_df.copy()

# ---- Time features ----
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Rush hour: 8-10 AM and 5-8 PM
df['is_rush_hour'] = (
    ((df['hour'] >= 8) & (df['hour'] < 10)) |  # Morning rush
    ((df['hour'] >= 17) & (df['hour'] < 20))    # Evening rush
).astype(int)

# Cyclical time encoding (helps ML understand that 23:00 is close to 00:00)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

print(f'Time features added')
print(f'Rush hour readings: {df["is_rush_hour"].sum()} / {len(df)}')
print(f'Weekend readings: {df["is_weekend"].sum()} / {len(df)}')

Time features added
Rush hour readings: 288 / 289
Weekend readings: 289 / 289


In [4]:
# ---- Location encoding ----
le_location = LabelEncoder()
df['location_encoded'] = le_location.fit_transform(df['location_name'])

print('Location encoding:')
for i, name in enumerate(le_location.classes_):
    count = (df['location_encoded'] == i).sum()
    print(f'  {i} → {name} ({count} readings)')

Location encoding:
  0 → Sarjapur Road Junction (1 readings)
  1 → carmelaram_junction (36 readings)
  2 → chembenahalli (36 readings)
  3 → dommasandra_circle (36 readings)
  4 → harlur_road_junction (36 readings)
  5 → iblur_wipro_junction (36 readings)
  6 → sarjapur_road_junction (36 readings)
  7 → sarjapur_town_center (36 readings)
  8 → varthur_gunjur_junction (36 readings)


In [5]:
# ---- Weather features (merge nearest weather reading) ----
# For each traffic reading, find the closest weather reading in time

if len(weather_df) > 0:
    weather_features = weather_df[['timestamp', 'temperature', 'humidity', 
                                    'wind_speed', 'rain_1h', 'visibility']].copy()
    weather_features = weather_features.sort_values('timestamp')
    
    # Use merge_asof to join by nearest timestamp
    df = df.sort_values('timestamp')
    df = pd.merge_asof(
        df,
        weather_features,
        on='timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('2h'),  # Max 2h gap
    )
    
    # Fill any NaN weather with defaults
    df['temperature'] = df['temperature'].fillna(25.0)
    df['humidity'] = df['humidity'].fillna(60.0)
    df['wind_speed'] = df['wind_speed'].fillna(2.0)
    df['rain_1h'] = df['rain_1h'].fillna(0.0)
    df['visibility'] = df['visibility'].fillna(10000.0)
    
    # Is it raining?
    df['is_raining'] = (df['rain_1h'] > 0).astype(int)
    
    print(f'Weather features merged')
    print(f'Raining readings: {df["is_raining"].sum()}')
else:
    # No weather data — add defaults
    df['temperature'] = 25.0
    df['humidity'] = 60.0
    df['wind_speed'] = 2.0
    df['rain_1h'] = 0.0
    df['visibility'] = 10000.0
    df['is_raining'] = 0
    print('⚠️ No weather data — using defaults')

Weather features merged
Raining readings: 0


In [6]:
# ---- Lag features (speed history for each location) ----
# Sort by location and time
df = df.sort_values(['location_name', 'timestamp']).reset_index(drop=True)

# Create lag features per location
for lag_name, lag_periods in [('lag_1', 1), ('lag_3', 3), ('lag_6', 6)]:
    df[f'speed_{lag_name}'] = df.groupby('location_name')['current_speed'].shift(lag_periods)
    df[f'ratio_{lag_name}'] = df.groupby('location_name')['congestion_ratio'].shift(lag_periods)

# Rolling averages
df['speed_rolling_3'] = df.groupby('location_name')['current_speed'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df['speed_rolling_6'] = df.groupby('location_name')['current_speed'].transform(
    lambda x: x.rolling(6, min_periods=1).mean()
)

# Drop rows with NaN lags (first few readings per location)
before = len(df)
df = df.dropna(subset=['speed_lag_1', 'speed_lag_3', 'speed_lag_6'])
after = len(df)

print(f'Lag features added')
print(f'Dropped {before - after} rows with NaN lags')
print(f'Final dataset: {len(df):,} rows')

Lag features added
Dropped 49 rows with NaN lags
Final dataset: 240 rows


## 3. Prepare Train/Test Split (Time-Based)

In [7]:
# ---- Define features and target ----
feature_cols = [
    # Time features
    'hour', 'day_of_week', 'is_weekend', 'is_rush_hour',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    # Location
    'location_encoded',
    # Traffic context
    'free_flow_speed', 'confidence',
    # Weather
    'temperature', 'humidity', 'wind_speed', 'rain_1h', 'visibility', 'is_raining',
    # Lag features
    'speed_lag_1', 'speed_lag_3', 'speed_lag_6',
    'ratio_lag_1', 'ratio_lag_3', 'ratio_lag_6',
    'speed_rolling_3', 'speed_rolling_6',
]

target_col = 'current_speed'

# Sort by time for proper time-based split
df = df.sort_values('timestamp').reset_index(drop=True)

X = df[feature_cols]
y = df[target_col]

# 80/20 split by time (NO shuffle!)
split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Features: {len(feature_cols)}')
print(f'Train: {len(X_train):,} rows ({df["timestamp"].iloc[0]} → {df["timestamp"].iloc[split_idx-1]})')
print(f'Test:  {len(X_test):,} rows ({df["timestamp"].iloc[split_idx]} → {df["timestamp"].iloc[-1]})')
print(f'\nTarget stats (train):')
print(f'  Mean: {y_train.mean():.1f} km/h')
print(f'  Std:  {y_train.std():.1f} km/h')
print(f'  Min:  {y_train.min():.1f} km/h')
print(f'  Max:  {y_train.max():.1f} km/h')

Features: 25
Train: 192 rows (2026-04-18 17:41:11.488254 → 2026-04-18 18:38:30.851847)
Test:  48 rows (2026-04-18 18:41:11.425599 → 2026-04-18 18:53:30.701281)

Target stats (train):
  Mean: 29.8 km/h
  Std:  5.1 km/h
  Min:  23.0 km/h
  Max:  36.0 km/h


## 4. Train Random Forest

In [8]:
# ---- Random Forest ----
print('Training Random Forest...')

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

# Predict
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

# Evaluate
print(f'\n📊 Random Forest Results:')
print(f'  TRAIN — MAE: {mean_absolute_error(y_train, rf_train_pred):.2f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_train, rf_train_pred)):.2f} | '
      f'R²: {r2_score(y_train, rf_train_pred):.4f}')
print(f'  TEST  — MAE: {mean_absolute_error(y_test, rf_test_pred):.2f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_test, rf_test_pred)):.2f} | '
      f'R²: {r2_score(y_test, rf_test_pred):.4f}')

Training Random Forest...

📊 Random Forest Results:
  TRAIN — MAE: 0.02 | RMSE: 0.10 | R²: 0.9996
  TEST  — MAE: 0.02 | RMSE: 0.07 | R²: 0.9998


## 5. Train Gradient Boosting

In [9]:
# ---- Gradient Boosting ----
print('Training Gradient Boosting...')

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.8,
    random_state=42,
)

gb_model.fit(X_train, y_train)

# Predict
gb_train_pred = gb_model.predict(X_train)
gb_test_pred = gb_model.predict(X_test)

# Evaluate
print(f'\n📊 Gradient Boosting Results:')
print(f'  TRAIN — MAE: {mean_absolute_error(y_train, gb_train_pred):.2f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_train, gb_train_pred)):.2f} | '
      f'R²: {r2_score(y_train, gb_train_pred):.4f}')
print(f'  TEST  — MAE: {mean_absolute_error(y_test, gb_test_pred):.2f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_test, gb_test_pred)):.2f} | '
      f'R²: {r2_score(y_test, gb_test_pred):.4f}')

Training Gradient Boosting...

📊 Gradient Boosting Results:
  TRAIN — MAE: 0.00 | RMSE: 0.00 | R²: 1.0000
  TEST  — MAE: 0.02 | RMSE: 0.08 | R²: 0.9997


## 6. Compare Models & Select Best

In [10]:
# Compare
rf_mae = mean_absolute_error(y_test, rf_test_pred)
gb_mae = mean_absolute_error(y_test, gb_test_pred)
rf_r2 = r2_score(y_test, rf_test_pred)
gb_r2 = r2_score(y_test, gb_test_pred)

print('=' * 55)
print('MODEL COMPARISON (Test Set)')
print('=' * 55)
print(f'{"Model":<25s} {"MAE":>8s} {"RMSE":>8s} {"R²":>8s}')
print(f'{"-"*25} {"-"*8} {"-"*8} {"-"*8}')
print(f'{"Random Forest":<25s} {rf_mae:>8.2f} {np.sqrt(mean_squared_error(y_test, rf_test_pred)):>8.2f} {rf_r2:>8.4f}')
print(f'{"Gradient Boosting":<25s} {gb_mae:>8.2f} {np.sqrt(mean_squared_error(y_test, gb_test_pred)):>8.2f} {gb_r2:>8.4f}')

# Select best model
if gb_r2 > rf_r2:
    best_model = gb_model
    best_name = 'GradientBoosting'
    best_pred = gb_test_pred
else:
    best_model = rf_model
    best_name = 'RandomForest'
    best_pred = rf_test_pred

print(f'\n🏆 Best model: {best_name}')

MODEL COMPARISON (Test Set)
Model                          MAE     RMSE       R²
------------------------- -------- -------- --------
Random Forest                 0.02     0.07   0.9998
Gradient Boosting             0.02     0.08   0.9997

🏆 Best model: RandomForest


## 7. Feature Importances

In [11]:
# Feature importances from best model
importances = best_model.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=True)

fig = px.bar(
    feat_imp,
    x='importance',
    y='feature',
    orientation='h',
    title=f'🌟 Feature Importances ({best_name})',
    labels={'importance': 'Importance', 'feature': 'Feature'},
)
fig.update_layout(height=600)
fig.show()

print('\nTop 10 Features:')
for _, row in feat_imp.tail(10).iloc[::-1].iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f'  {row["feature"]:<25s} {row["importance"]:.4f} {bar}')


Top 10 Features:
  speed_lag_6               0.2187 █████████████████████
  speed_rolling_6           0.1940 ███████████████████
  speed_lag_3               0.1593 ███████████████
  free_flow_speed           0.1571 ███████████████
  speed_lag_1               0.1368 █████████████
  speed_rolling_3           0.1167 ███████████
  location_encoded          0.0173 █
  ratio_lag_3               0.0000 
  confidence                0.0000 
  ratio_lag_6               0.0000 


## 8. Actual vs Predicted Plot

In [12]:
# Scatter plot: actual vs predicted
fig = px.scatter(
    x=y_test.values,
    y=best_pred,
    labels={'x': 'Actual Speed (km/h)', 'y': 'Predicted Speed (km/h)'},
    title=f'📊 Actual vs Predicted ({best_name})',
    opacity=0.5,
)
# Add perfect prediction line
min_val = min(y_test.min(), best_pred.min())
max_val = max(y_test.max(), best_pred.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', name='Perfect prediction',
    line=dict(color='red', dash='dash')
))
fig.update_layout(height=500)
fig.show()

## 9. Save Model & Artifacts

In [13]:
# Create models directory
models_dir = Path.cwd().parent / 'models'
models_dir.mkdir(exist_ok=True)

# Save the best model
model_path = models_dir / 'traffic_speed_model.joblib'
joblib.dump(best_model, model_path)
print(f'✅ Model saved: {model_path}')

# Save the label encoder
le_path = models_dir / 'location_encoder.joblib'
joblib.dump(le_location, le_path)
print(f'✅ Encoder saved: {le_path}')

# Save feature columns list
config_path = models_dir / 'model_config.joblib'
model_config = {
    'feature_cols': feature_cols,
    'target_col': target_col,
    'model_name': best_name,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'test_mae': mean_absolute_error(y_test, best_pred),
    'test_rmse': np.sqrt(mean_squared_error(y_test, best_pred)),
    'test_r2': r2_score(y_test, best_pred),
    'location_classes': list(le_location.classes_),
}
joblib.dump(model_config, config_path)
print(f'✅ Config saved: {config_path}')

print(f'\n📋 Model Summary:')
for k, v in model_config.items():
    if k != 'feature_cols' and k != 'location_classes':
        print(f'  {k}: {v}')

✅ Model saved: d:\GitHub_Works\Urban Intelligence Dashboard\models\traffic_speed_model.joblib
✅ Encoder saved: d:\GitHub_Works\Urban Intelligence Dashboard\models\location_encoder.joblib
✅ Config saved: d:\GitHub_Works\Urban Intelligence Dashboard\models\model_config.joblib

📋 Model Summary:
  target_col: current_speed
  model_name: RandomForest
  train_size: 192
  test_size: 48
  test_mae: 0.01751537698412668
  test_rmse: 0.07450514014782135
  test_r2: 0.9997844120313354
